# ExpW: RGB224 + PCA64 + RBF SVM

Run the numbered cells from top to bottom. Steps 1–4 set up the experiment; steps 5–16 define the individual operations; step 17 runs training and testing; steps 18–20 display and export the results.

**QUICK TEST** It uses 20 training and 5 test images per class in outer fold 1. For the full protocol, set `QUCIK_TEST=False` and `FOLDS_TO_RUN=list(range(1, 11))`.

Train and tune on clean images only. Scaling and 64-component IncrementalPCA are fitted separately within each training split, using batches rather than a full image matrix.


## 1. Import libraries

In [132]:
# Run from the folder containing the notebook, metadata, splits and origin/ images.
from pathlib import Path
import hashlib
import json
import tempfile
import time
import platform
import zipfile
from datetime import datetime, timezone

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
import scipy
from scipy.ndimage import gaussian_filter
import sklearn
from sklearn.decomposition import IncrementalPCA
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from threadpoolctl import threadpool_limits
from IPython.display import display


In [133]:
import warnings

warnings.filterwarnings(
    "ignore",
    message=r"(divide by zero|overflow|invalid value) encountered in matmul",
    category=RuntimeWarning,
    module=r"sklearn\.decomposition\._base",
)

## 2. Set experiment options

Set the paths relative to the current working directory. Images are read from `./origin`; metadata and splits are loaded from the two files beside the notebook. Change the QUICK/full-run settings here. Set `RUN_LEARNING_CURVES=True` for additional training-size experiments.


In [134]:
PROJECT_DIR = Path.cwd()
IMAGE_ROOT = PROJECT_DIR / "images"/ "origin"
METADATA_CSV = PROJECT_DIR / "expw_subset_metadata_comp90051.csv"
SPLITS_JSON = PROJECT_DIR / "Outer_10-fold_CV_splits.json"
RESULTS_ROOT = PROJECT_DIR / "expw_svm_results"
SEED = 90051

QUICK_TEST = False  # Small real-data run; NOT the project's final performance estimate.
FOLDS_TO_RUN = list(range(1, 11))  # Final run: QUICK_TEST=False; list(range(1, 11)).
TRAIN_PER_CLASS = 20
TEST_PER_CLASS = 5
IMAGE_SIZE = (128, 128)
MASK_FILL = 128  # Grey in all three RGB channels.
PCA_COMPONENTS = 64
BATCH_SIZE = 128
RUN_LEARNING_CURVES = True  # Extra fits: enable when you need learning curves.
LEARNING_CURVE_FRACTIONS = [0.1, 0.25, 0.5, 0.75, 1.0]
SAVE_PCA_SCORES = True  # Small 64-dimensional arrays, not full image vectors.
CONDITIONS = ["clean", "upper_mask", "lower_mask", "gaussian_blur", "gaussian_noise"]
C_VALUES = [0.1, 1.0, 10.0]
GAMMA_MULTIPLIERS = [0.1, 1.0, 10.0]
REFERENCE_HASH = "deb6fa9592d87248968d7c5b67f1867ad5b589c8f2dfe3aa3ec450917461dfec"
EXPECTED_COUNT = 14935


## 3. Load and verify the local data and splits

Load the local CSV and JSON, then check the metadata fingerprint, class labels, image directory and non-overlapping outer splits. Keep the metadata row order unchanged.


In [135]:
metadata = pd.read_csv(METADATA_CSV)
y = metadata["correct_facial_class"].to_numpy(dtype=int)

with SPLITS_JSON.open(encoding="utf-8") as f:
    outer_folds = json.load(f)

# Recorded in Section 4; these do not perform validation.
actual_hash = hashlib.sha256(
    metadata.to_csv(index=False, lineterminator="\n").encode()
).hexdigest()
splits_hash = hashlib.sha256(SPLITS_JSON.read_bytes()).hexdigest()

## 4. Create the results folder and record settings

Each run gets its own folder. Save settings, package versions, metadata and exact outer splits for reproducibility.


In [136]:
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
RUN_DIR = Path(tempfile.mkdtemp(prefix="rgb224_pca64_rbf_", dir=RESULTS_ROOT))
config = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "python_version": platform.python_version(), "results_schema": 2,
    "project_dir": str(PROJECT_DIR), "metadata_file": str(METADATA_CSV),
    "splits_file": str(SPLITS_JSON), "metadata_hash": actual_hash, "splits_sha256": splits_hash,
    "seed": SEED, "QUICK_TEST": QUICK_TEST, "folds": FOLDS_TO_RUN,
    "QUICK_train_per_class": TRAIN_PER_CLASS, "QUICK_TEST_per_class": TEST_PER_CLASS,
    "image_root": str(IMAGE_ROOT), "image_size": IMAGE_SIZE, "color": "RGB",
    "pixel_scale": "float32 pixels / 255", "resize": "Pillow BILINEAR",
    "pca": "IncrementalPCA: approximate, fitted on training batches only",
    "pca_components": PCA_COMPONENTS, "batch_size": BATCH_SIZE,
    "save_pca_scores": SAVE_PCA_SCORES,
    "learning_curves": RUN_LEARNING_CURVES,
    "learning_curve_fractions": LEARNING_CURVE_FRACTIONS,
    "C_values": C_VALUES, "gamma_multipliers": GAMMA_MULTIPLIERS,
    "gamma_reference": "1 / (64 * variance of current training PCA scores)",
    "class_weight": "balanced", "selection_metric": "macro_f1", "inner_folds": 3,
    "training": "clean only", "conditions": CONDITIONS,
    "operation_order": "bbox crop -> RGB -> resize224 -> independent corruption",
    "mask_fill": MASK_FILL, "mask_boundary": "height // 2", "blur_sigma": 1.0,
    "blur_kernel": [7, 7], "blur_axes": [0, 1], "blur_boundary": "reflect",
    "noise_sd": 0.05, "noise_seed_rule": "SeedSequence([90051, original_metadata_row])",
    "noise_channels": "independent RGB", "PNG_storage": "clip then round to uint8",
    "numpy_version": np.__version__, "scipy_version": scipy.__version__,
    "sklearn_version": sklearn.__version__,
    "class_codes": list(range(7)), "zero_division": 0,
    "decision_scores": "SVC OVR scores in classes_ order; NOT probabilities",
    "regression_metrics": "MAE/RMSE omitted: emotion codes are unordered categories",
}
(RUN_DIR / "config.json").write_text(json.dumps(config, indent=2))
(RUN_DIR / "outer_splits.json").write_text(json.dumps(outer_folds, indent=2))
metadata.to_csv(RUN_DIR / "metadata.csv", index=False)
print("QUICK TEST — not a final performance estimate" if QUICK_TEST else "FULL-DATA RUN")
print("Local output directory:", RUN_DIR)


FULL-DATA RUN
Local output directory: /Users/fr/Desktop/COMP90051/expw_svm_results/rgb224_pca64_rbf_xjgu08q6


## 5. Define RGB preprocessing and corruptions

Define the crop, resize, grey masks, blur and noise together. The row-specific noise seed makes the corruption independent of processing order.


In [137]:
def crop_face(metadata_row):
    # Same crop/RGB/resize order as shared_preprocessing.ipynb; no EXIF rotation.
    row = metadata.iloc[int(metadata_row)]
    filename = str(row["original_image_name"])
    if Path(filename).name != filename:
        raise ValueError(f"Expected a plain original filename: {filename!r}")
    with Image.open(IMAGE_ROOT / filename) as source:
        bounds = np.asarray([row[k] for k in ["left", "top", "right", "bottom"]], dtype=float)
        if not np.isfinite(bounds).all() or not np.equal(bounds, np.floor(bounds)).all():
            raise ValueError(f"Non-integer crop coordinates at row {metadata_row}")
        left, top, right, bottom = map(int, bounds)
        if not (0 <= left < right <= source.width and 0 <= top < bottom <= source.height):
            raise ValueError(f"Crop outside source image at row {metadata_row}")
        if min(right - left, bottom - top) < 100:
            raise ValueError(f"Crop smaller than 100 pixels at row {metadata_row}")
        face = source.convert("RGB").crop((left, top, right, bottom))
    return face.resize(IMAGE_SIZE, Image.Resampling.BILINEAR)


def make_variants(face, metadata_row):
    # Each variant starts from the same clean224 RGB image, never another corruption.
    upper, lower = face.copy(), face.copy()
    half = face.height // 2
    fill = (MASK_FILL,) * 3
    upper.paste(fill, (0, 0, face.width, half))
    lower.paste(fill, (0, half, face.width, face.height))
    pixels = np.asarray(face, dtype=np.float32) / 255.0
    blurred = gaussian_filter(pixels, sigma=1.0, radius=3, axes=(0, 1))
    # Identical row-specific RGB noise for all three methods, independent of order.
    rng = np.random.default_rng(np.random.SeedSequence([90051, int(metadata_row)]))
    noisy = pixels + rng.normal(0.0, 0.05, size=pixels.shape)
    def to_png(values):
        return Image.fromarray(np.rint(np.clip(values, 0.0, 1.0) * 255).astype(np.uint8))
    return {"clean": face.copy(), "upper_mask": upper, "lower_mask": lower,
            "gaussian_blur": to_png(blurred), "gaussian_noise": to_png(noisy)}


## 6. Load RGB images in small batches

Read only the current batch into memory. Each RGB image becomes a vector of 150,528 scaled pixel values.


In [138]:
def batches(indices, minimum=1):
    indices = np.asarray(indices, dtype=int)
    if len(indices) < minimum:
        raise ValueError(f"Need at least {minimum} rows, got {len(indices)}.")
    chunks = [indices[i:i + BATCH_SIZE] for i in range(0, len(indices), BATCH_SIZE)]
    if len(chunks) > 1 and len(chunks[-1]) < minimum:
        chunks[-2] = np.r_[chunks[-2], chunks[-1]]
        chunks.pop()
    return chunks


def load_batch(indices, paths=None):
    # No full N x150528 array: only the current RGB batch is in memory.
    values = np.empty((len(indices), IMAGE_SIZE[0] * IMAGE_SIZE[1] * 3), dtype=np.float32)
    for position, metadata_row in enumerate(indices):
        if paths is None:
            face = crop_face(metadata_row)
        else:
            with Image.open(paths[int(metadata_row)]) as saved:
                if saved.mode != "RGB" or saved.size != IMAGE_SIZE:
                    raise ValueError("Expected a shared224 RGB PNG, with no further resize.")
                face = saved.copy()
        values[position] = np.asarray(face, dtype=np.float32).ravel() / 255.0
    return values


## 7. Fit and apply batch PCA64

Fit the scaler first, then IncrementalPCA with 64 components. Both see only the current training split; validation and test images are transformed without refitting.


In [139]:
def fit_preprocessor(train_indices):
    # Fit scaling and approximate PCA on this training split only.
    order = np.random.default_rng(SEED).permutation(train_indices)
    scaler = StandardScaler()
    for batch in batches(order):
        scaler.partial_fit(load_batch(batch))
    pca = IncrementalPCA(n_components=PCA_COMPONENTS, batch_size=BATCH_SIZE)
    for batch in batches(order, minimum=PCA_COMPONENTS):
        pca.partial_fit(scaler.transform(load_batch(batch)))
    return scaler, pca


def transform(indices, scaler, pca, paths=None):
    scores = np.empty((len(indices), PCA_COMPONENTS), dtype=np.float32)
    position = 0
    for batch in batches(indices):
        scores[position:position + len(batch)] = pca.transform(scaler.transform(load_batch(batch, paths)))
        position += len(batch)
    return scores


## 8. Save identical test images for all methods

Save the five RGB versions and their manifest so the other methods can use exactly the same test inputs.


In [140]:
def save_test_variants(test_indices, fold_dir):
    for condition in CONDITIONS:
        (fold_dir / "shared_variants" / condition).mkdir(parents=True, exist_ok=True)
    records = []
    for metadata_row in test_indices:
        metadata_row = int(metadata_row)
        row = metadata.iloc[metadata_row]
        record = {"metadata_row": metadata_row, "sample_face_id": row["sample_face_id"],
                  "original_image_name": row["original_image_name"],
                  "correct_facial_class": int(y[metadata_row]),
                  "output_width": IMAGE_SIZE[0], "output_height": IMAGE_SIZE[1]}
        for condition, face in make_variants(crop_face(metadata_row), metadata_row).items():
            relative = Path("shared_variants") / condition / f"{metadata_row:05d}.png"
            face.save(fold_dir / relative, format="PNG")
            record[condition + "_path"] = relative.as_posix()
        records.append(record)
    manifest = pd.DataFrame(records)
    manifest.to_csv(fold_dir / "shared_test_manifest.csv", index=False)
    return manifest


## 9. Create training and validation splits

Use the fixed outer splits, and create three stratified inner splits for parameter selection. QUICK-test sampling is reproducible.


In [141]:
def sample_per_class(indices, count, seed):
    rng = np.random.default_rng(seed)
    chosen = []
    for label in range(7):
        available = indices[y[indices] == label]
        if len(available) < count:
            raise ValueError(f"Class {label}: need {count} samples, found {len(available)}.")
        chosen.extend(rng.choice(available, count, replace=False))
    return np.asarray(sorted(chosen), dtype=int)


def make_inner_folds(train_indices):
    rng = np.random.default_rng(SEED)
    validation = [[], [], []]
    start = 0
    for label in range(7):
        indices = train_indices[y[train_indices] == label].copy()
        rng.shuffle(indices)
        for position, index in enumerate(indices):
            validation[(start + position) % 3].append(int(index))
        start = (start + len(indices)) % 3
    return [(np.setdiff1d(train_indices, val), np.asarray(sorted(val), dtype=int))
            for val in validation]


## 10. Calculate classification metrics

Calculate accuracy, balanced accuracy, precision, recall, F1 and confusion counts. MAE/RMSE on unordered emotion IDs are not meaningful classification measures.


In [142]:
def metrics(truth, prediction):
    # Seven-class metrics, with undefined precision/recall/F1 set to zero.
    truth, prediction = np.asarray(truth, dtype=int), np.asarray(prediction, dtype=int)
    if truth.ndim != 1 or truth.shape != prediction.shape or not len(truth):
        raise ValueError("Expected nonempty matching one-dimensional label arrays.")
    if not (np.isin(truth, range(7)).all() and np.isin(prediction, range(7)).all()):
        raise ValueError("Emotion codes must be in 0..6.")
    confusion = np.zeros((7, 7), dtype=int)
    np.add.at(confusion, (truth, prediction), 1)
    correct = np.diag(confusion)
    support = confusion.sum(axis=1)
    predicted = confusion.sum(axis=0)
    precision = np.divide(correct, predicted, out=np.zeros(7), where=predicted > 0)
    recall = np.divide(correct, support, out=np.zeros(7), where=support > 0)
    f1 = np.divide(2 * precision * recall, precision + recall,
                   out=np.zeros(7), where=precision + recall > 0)
    accuracy = float(correct.sum() / confusion.sum())
    overall = {"n_samples": int(len(truth)), "accuracy": accuracy,
               "balanced_accuracy": float(recall[support > 0].mean()),
               "macro_precision": float(precision.mean()), "macro_recall": float(recall.mean()),
               "macro_f1": float(f1.mean()),
               "weighted_precision": float(np.average(precision, weights=support)),
               "weighted_recall": float(np.average(recall, weights=support)),
               "weighted_f1": float(np.average(f1, weights=support)),
               "micro_precision": accuracy, "micro_recall": accuracy, "micro_f1": accuracy}
    per_class = pd.DataFrame({"class": range(7), "support": support,
                              "predicted_count": predicted, "tp": correct,
                              "fp": predicted - correct, "fn": support - correct,
                              "precision": precision, "recall": recall, "f1": f1})
    return overall, per_class, confusion


## 11. Save predictions, scores and reports

Save every test image's ID, true label, prediction, seven decision scores and optional PCA64 features, plus per-class and aggregate reports. SVM decision scores are not probabilities. The compact ZIP excludes large image/model files; those remain in the run folder.


In [143]:
def save_metric_reports(truth, prediction, folder):
    folder.mkdir(parents=True, exist_ok=True)
    overall, per_class, confusion = metrics(truth, prediction)
    (folder / "metrics.json").write_text(json.dumps(overall, indent=2))
    per_class.to_csv(folder / "per_class.csv", index=False)
    table = pd.DataFrame(confusion, index=range(7), columns=range(7))
    table.index.name, table.columns.name = "true_class", "predicted_class"
    table.to_csv(folder / "confusion.csv")
    support = confusion.sum(axis=1, keepdims=True)
    normalized = np.divide(confusion, support, out=np.zeros((7, 7)), where=support > 0)
    pd.DataFrame(normalized, index=table.index, columns=table.columns).to_csv(
        folder / "confusion_normalized.csv")
    return overall


def save_evaluation(indices, z, svm, folder, context):
    # Save sample-level evidence so later classification metrics need no refit.
    indices = np.asarray(indices, dtype=int)
    prediction = svm.predict(z)  # Do not replace predict() with argmax(decision_function()).
    decision = svm.decision_function(z)
    classes = svm.classes_.astype(int)
    if not np.array_equal(classes, np.arange(7)) or decision.shape != (len(indices), 7):
        raise ValueError("Expected seven OVR decision-score columns in class order.")
    if not np.isfinite(decision).all():
        raise ValueError("Non-finite SVM decision scores.")
    table = metadata.iloc[indices][["sample_face_id", "original_image_name"]].reset_index(drop=True)
    table.insert(0, "metadata_row", indices)
    table["y_true"], table["y_pred"] = y[indices], prediction
    table["correct"] = prediction == y[indices]
    for name, value in context.items():
        table[name] = value
    for column, label in enumerate(classes):
        table[f"decision_class_{label}"] = decision[:, column]
    folder.mkdir(parents=True, exist_ok=True)
    table.to_csv(folder / "predictions.csv", index=False)
    (folder / "evaluation.json").write_text(json.dumps({
        **context, "classes": classes.tolist(), "score_type": "uncalibrated OVR decision score",
        "svm_parameters": svm.get_params()}, indent=2))
    if SAVE_PCA_SCORES:
        np.savez_compressed(folder / "features.npz", metadata_row=indices,
                            y_true=y[indices], pca_scores=z)
    return save_metric_reports(y[indices], prediction, folder)


def archive_results():
    # Checkpoint a compact archive; large images/models stay in the run folder.
    archive_path = RUN_DIR.parent / (RUN_DIR.name + "_results.zip")
    temporary = archive_path.with_suffix(".zip.tmp")
    with zipfile.ZipFile(temporary, "w", zipfile.ZIP_DEFLATED) as archive:
        for path in sorted(RUN_DIR.rglob("*")):
            if path.is_file() and "shared_variants" not in path.parts and path.suffix != ".joblib":
                archive.write(path, path.relative_to(RUN_DIR))
    temporary.replace(archive_path)
    return archive_path


## 12. Define optional learning curves

Optional extra fits at increasing training sizes. Refit preprocessing for each subset and save training/validation predictions, metrics, timings and blue/red curves. These are diagnostic curves using selected hyperparameters, not independent test estimates. Subsets with fewer than 64 images are skipped.


In [144]:
def run_learning_curves(train_idx, fold_dir, best):
    # Diagnostic learning curves using only the current outer-training split.
    train_idx = np.asarray(train_idx, dtype=int)
    fold_dir = Path(fold_dir)
    number = int(fold_dir.name.removeprefix("fold_"))
    curve_dir = fold_dir / "learning_curves"
    curve_dir.mkdir(parents=True, exist_ok=True)
    fractions = sorted(set(float(f) for f in LEARNING_CURVE_FRACTIONS))
    if not fractions or not all(0 < f <= 1 for f in fractions):
        raise ValueError("Learning-curve fractions must be in (0, 1].")
    if len(np.unique(train_idx)) != len(train_idx):
        raise ValueError("Learning-curve training indices must be unique.")
    selected_c = float(best["C"])
    selected_multiplier = float(best["gamma_multiplier"])
    (curve_dir / "settings.json").write_text(json.dumps({
        "outer_fold": number, "fractions": fractions, "seed": SEED,
        "C": selected_c, "gamma_multiplier": selected_multiplier,
        "pca_components": PCA_COMPONENTS, "condition": "clean",
        "subset_rule": "Nested, seeded per-class prefixes; ceil(fraction * class count).",
        "purpose": "Diagnostic only: selected hyperparameters reuse inner-CV validation data, "
                   "so validation scores can have selection optimism. Outer-test data are unused.",
        "timing": "Fit times exclude evaluation; evaluation times include prediction and export.",
    }, indent=2))
    points = []
    for inner, (fit_idx, val_idx) in enumerate(make_inner_folds(train_idx), start=1):
        fit_idx = np.asarray(fit_idx, dtype=int)
        val_idx = np.asarray(val_idx, dtype=int)
        assert np.intersect1d(fit_idx, val_idx).size == 0
        assert np.isin(np.r_[fit_idx, val_idx], train_idx).all()
        assert np.array_equal(np.sort(np.r_[fit_idx, val_idx]), np.sort(train_idx))
        rng = np.random.default_rng(np.random.SeedSequence([SEED, number, inner]))
        class_order = {label: rng.permutation(fit_idx[y[fit_idx] == label])
                       for label in range(7)}
        previous_subset = np.array([], dtype=int)
        for point, fraction in enumerate(fractions, start=1):
            subset = np.sort(np.concatenate([
                indices[:int(np.ceil(fraction * len(indices)))]
                for indices in class_order.values()
            ])).astype(int)
            assert np.isin(previous_subset, subset).all()
            previous_subset = subset
            point_dir = curve_dir / f"inner_{inner}" / f"size_{point:02d}"
            point_dir.mkdir(parents=True, exist_ok=True)
            (point_dir / "indices.json").write_text(json.dumps({
                "training_fraction": fraction, "train": subset.tolist(),
                "validation": val_idx.tolist(), "inner_training_pool": fit_idx.tolist(),
            }, indent=2))
            result = {"outer_fold": number, "inner_fold": inner,
                      "training_fraction": fraction, "n_train": len(subset),
                      "n_validation": len(val_idx), "C": selected_c,
                      "gamma_multiplier": selected_multiplier}
            missing = sorted(set(range(7)) - set(y[subset].tolist()))
            if len(subset) < PCA_COMPONENTS or missing:
                reasons = []
                if len(subset) < PCA_COMPONENTS:
                    reasons.append(f"Need at least {PCA_COMPONENTS} training rows for PCA64")
                if missing:
                    reasons.append(f"Missing training classes: {missing}")
                result.update(status="skipped", reason="; ".join(reasons))
                (point_dir / "status.json").write_text(json.dumps(result, indent=2))
                points.append(result)
                pd.DataFrame(points).to_csv(curve_dir / "results.csv", index=False)
                print(f"  Learning curve inner {inner}, fraction {fraction:g}: "
                      f"skipped ({result['reason']})", flush=True)
                continue
            print(f"  Learning curve inner {inner}, fraction {fraction:g}: "
                  f"{len(subset)} clean training faces", flush=True)
            started = time.perf_counter()
            try:
                fit_started = time.perf_counter()
                scaler, pca = fit_preprocessor(subset)
                result["preprocessor_fit_seconds"] = time.perf_counter() - fit_started
                transform_started = time.perf_counter()
                z_train = transform(subset, scaler, pca)
                z_val = transform(val_idx, scaler, pca)
                result["transform_seconds"] = time.perf_counter() - transform_started
                gamma = float(selected_multiplier * gamma_reference(z_train))
                result["gamma"] = gamma
                svm = SVC(C=selected_c, gamma=gamma, kernel="rbf",
                          class_weight="balanced", cache_size=256)
                fit_started = time.perf_counter()
                svm.fit(z_train, y[subset])
                result["svm_fit_seconds"] = time.perf_counter() - fit_started
                result["fit_seconds"] = (result["preprocessor_fit_seconds"]
                                         + result["svm_fit_seconds"])
                context = {"outer_fold": number, "inner_fold": inner,
                           "training_fraction": fraction, "n_train": len(subset),
                           "condition": "clean", "C": selected_c,
                           "gamma_multiplier": selected_multiplier, "gamma": gamma}
                for name, indices, values in [("train", subset, z_train),
                                              ("validation", val_idx, z_val)]:
                    evaluation_started = time.perf_counter()
                    overall = save_evaluation(indices, values, svm, point_dir / name,
                                              {**context, "evaluation": "learning_curve_" + name})
                    result[name + "_evaluation_seconds"] = time.perf_counter() - evaluation_started
                    result.update({name + "_" + key: value for key, value in overall.items()})
                result.update(status="complete", reason="")
                result["total_seconds"] = time.perf_counter() - started
                del scaler, pca, z_train, z_val, svm
            except Exception as error:
                result.update(status="failed", reason=f"{type(error).__name__}: {error}")
                result["total_seconds"] = time.perf_counter() - started
                (point_dir / "status.json").write_text(json.dumps(result, indent=2))
                points.append(result)
                pd.DataFrame(points).to_csv(curve_dir / "results.csv", index=False)
                raise  # Preserve completed points, but never hide a failed computation.
            (point_dir / "status.json").write_text(json.dumps(result, indent=2))
            points.append(result)
            pd.DataFrame(points).to_csv(curve_dir / "results.csv", index=False)

    table = pd.DataFrame(points)
    complete = table[table["status"] == "complete"]
    if complete.empty:
        print("  No learning-curve point had enough samples for PCA64.", flush=True)
        return table
    columns = ["n_train", "n_validation", "fit_seconds", "total_seconds",
               "train_accuracy", "validation_accuracy", "train_macro_f1", "validation_macro_f1"]
    summary = complete.groupby("training_fraction")[columns].agg(["mean", "std"])
    summary.columns = ["_".join(column) for column in summary.columns]
    summary["n_inner_folds"] = complete.groupby("training_fraction").size()
    summary.to_csv(curve_dir / "summary.csv")
    for metric, label in [("accuracy", "Accuracy"), ("macro_f1", "Macro F1")]:
        figure, axis = plt.subplots(figsize=(6, 4), constrained_layout=True)
        for name, color in [("train", "blue"), ("validation", "red")]:
            values = summary[name + "_" + metric + "_mean"]
            deviation = summary[name + "_" + metric + "_std"].fillna(0)
            axis.plot(summary["n_train_mean"], values, marker="o", color=color, label=name)
            axis.fill_between(summary["n_train_mean"], values - deviation,
                              values + deviation, color=color, alpha=0.15)
        axis.set(xlabel="Training faces (mean across inner folds)", ylabel=label,
                 ylim=(0, 1), title=f"Fold {number}: diagnostic learning curve")
        axis.legend()
        figure.savefig(curve_dir / f"{metric}.png", dpi=160, bbox_inches="tight")
        plt.close(figure)
    return table


## 13. Tune the RBF SVM on inner validation splits

Try all nine C/gamma combinations in three inner folds. Save each candidate's validation predictions and scores, then select the highest mean macro F1.


In [145]:
def gamma_reference(scores):
    variance = np.var(scores, dtype=np.float64)
    if not np.isfinite(variance) or variance <= 0:
        raise ValueError("Training PCA scores must have positive finite variance.")
    return 1.0 / (PCA_COMPONENTS * variance)


def tune_parameters(train_idx, fold_dir, number):
    inner_results = []
    for inner, (fit_idx, val_idx) in enumerate(make_inner_folds(train_idx), start=1):
        inner_dir = fold_dir / "inner_cv" / f"split_{inner}"
        inner_dir.mkdir(parents=True)
        (inner_dir / "indices.json").write_text(json.dumps({
            "train": fit_idx.tolist(), "validation": val_idx.tolist()}, indent=2))
        print(f"  Inner {inner}/3: fitting train-only scaler/PCA64", flush=True)
        fit_start = time.perf_counter()
        scaler, pca = fit_preprocessor(fit_idx)
        z_train = transform(fit_idx, scaler, pca)
        z_val = transform(val_idx, scaler, pca)
        gamma0 = gamma_reference(z_train)
        preprocessing_seconds = time.perf_counter() - fit_start
        for c_index, c in enumerate(C_VALUES):
            for g_index, multiplier in enumerate(GAMMA_MULTIPLIERS):
                candidate_dir = inner_dir / f"candidate_{c_index}_{g_index}"
                svm = SVC(C=c, gamma=multiplier * gamma0, kernel="rbf", class_weight="balanced", cache_size=256,
                          decision_function_shape="ovr")
                fit_start = time.perf_counter()
                svm.fit(z_train, y[fit_idx])
                fit_seconds = time.perf_counter() - fit_start
                context = {"outer_fold": int(number), "inner_fold": inner, "condition": "clean",
                           "C": c, "gamma_multiplier": multiplier, "gamma": multiplier * gamma0}
                train_score = metrics(y[fit_idx], svm.predict(z_train))[0]
                score = save_evaluation(val_idx, z_val, svm, candidate_dir,
                                        {**context, "evaluation": "inner_validation"})
                inner_results.append({"inner_fold": inner, "C": c,
                                      "gamma_multiplier": multiplier, "gamma": multiplier * gamma0,
                                      "preprocessing_seconds": preprocessing_seconds,
                                      "svm_fit_seconds": fit_seconds, **score,
                                      **{"train_" + k: v for k, v in train_score.items()}})
                pd.DataFrame(inner_results).to_csv(fold_dir / "inner_cv_results.csv", index=False)
        del scaler, pca, z_train, z_val, svm
    inner_table = pd.DataFrame(inner_results)
    inner_table.to_csv(fold_dir / "inner_cv_results.csv", index=False)
    ranking = inner_table.groupby(["C", "gamma_multiplier"], as_index=False)["macro_f1"].mean()
    ranking = ranking.sort_values(["macro_f1", "C", "gamma_multiplier"], ascending=[False, True, True])
    ranking.to_csv(fold_dir / "inner_cv_means.csv", index=False)
    return ranking.iloc[0]


## 14. Fit the selected model on all outer-training data

Refit scaling, PCA64 and the selected RBF SVM using all outer-training images. Save the fitted model and training results.


In [146]:
def fit_final_model(train_idx, fold_dir, number, best):
    print("  Refit selected parameters on all outer-training faces", flush=True)
    scaler, pca = fit_preprocessor(train_idx)
    z_train = transform(train_idx, scaler, pca)
    gamma = float(best["gamma_multiplier"] * gamma_reference(z_train))
    svm = SVC(C=float(best["C"]), gamma=gamma, kernel="rbf", class_weight="balanced", cache_size=256,
              decision_function_shape="ovr")
    svm.fit(z_train, y[train_idx])
    joblib.dump({"scaler": scaler, "pca": pca, "svm": svm, "config": config}, fold_dir / "model.joblib")
    context = {"outer_fold": int(number), "C": float(best["C"]),
               "gamma_multiplier": float(best["gamma_multiplier"]), "gamma": gamma}
    save_evaluation(train_idx, z_train, svm, fold_dir / "train_clean",
                    {**context, "evaluation": "training", "condition": "clean"})
    return scaler, pca, svm, context


## 15. Evaluate the five held-out test conditions

Evaluate clean, upper-mask, lower-mask, blur and noise versions of the held-out faces. Save sample-level results and metrics separately for each condition.


In [147]:
def evaluate_test_conditions(test_idx, fold_dir, scaler, pca, svm, context):
    number = context["outer_fold"]
    manifest = save_test_variants(test_idx, fold_dir)
    rows = []
    for condition in CONDITIONS:
        paths = {int(row.metadata_row): fold_dir / getattr(row, condition + "_path")
                 for row in manifest.itertuples(index=False)}
        z_test = transform(test_idx, scaler, pca, paths)
        overall = save_evaluation(test_idx, z_test, svm, fold_dir / "test" / condition,
                                  {**context, "evaluation": "outer_test", "condition": condition})
        rows.append({"fold": number, "condition": condition, "n_test": len(test_idx), **overall})
        pd.DataFrame(rows).to_csv(fold_dir / "test_metrics.csv", index=False)
    return rows


## 16. Put the steps together for one outer fold

Call tuning, final fitting and testing in order for one outer fold. Test results are checkpointed before optional learning-curve fits.


In [148]:
def run_fold(fold):
    number = fold["fold"]
    train_idx = np.asarray(fold["train_indices"], dtype=int)
    test_idx = np.asarray(fold["test_indices"], dtype=int)
    if QUICK_TEST:
        train_idx = sample_per_class(train_idx, TRAIN_PER_CLASS, SEED + number)
        test_idx = sample_per_class(test_idx, TEST_PER_CLASS, SEED + 100 + number)
    fold_dir = RUN_DIR / f"fold_{number}"
    fold_dir.mkdir()
    (fold_dir / "used_indices.json").write_text(json.dumps({
        "train": train_idx.tolist(), "test": test_idx.tolist()}, indent=2))
    print(f"Fold {number}: {len(train_idx)} clean training; {len(test_idx)} held-out faces", flush=True)
    start = time.perf_counter()
    best = tune_parameters(train_idx, fold_dir, number)
    scaler, pca, svm, context = fit_final_model(train_idx, fold_dir, number, best)
    rows = evaluate_test_conditions(test_idx, fold_dir, scaler, pca, svm, context)
    elapsed = time.perf_counter() - start
    selection = {"fold": number, "C": float(best["C"]),
                 "gamma_multiplier": float(best["gamma_multiplier"]), "gamma": context["gamma"],
                 "inner_macro_f1": float(best["macro_f1"]), "seconds": elapsed}
    (fold_dir / "selected_parameters.json").write_text(json.dumps(selection, indent=2))
    archive_results()  # Test results are saved before optional extra fits.
    if RUN_LEARNING_CURVES:
        run_learning_curves(train_idx, fold_dir, best)
    else:
        (fold_dir / "learning_curves_status.json").write_text(json.dumps({
            "status": "not_requested", "enable_with": "RUN_LEARNING_CURVES = True"}))
    print(f"Fold {number} completed in {elapsed / 60:.1f} minutes", flush=True)
    # display(pd.DataFrame(rows).round(4))
    return rows, selection


## 17. Run the selected outer folds

**Training starts in this cell.** Execute the chosen folds with two CPU threads. Completed and partial results are saved if a later step fails.


In [149]:
results, selected_parameters = [], []
with threadpool_limits(limits=2):
    for fold in outer_folds:
        if fold["fold"] not in FOLDS_TO_RUN:
            continue
        try:
            rows, selection = run_fold(fold)
        except Exception as error:
            (RUN_DIR / "RUN_FAILED.json").write_text(json.dumps({
                "fold": fold["fold"], "error_type": type(error).__name__,
                "message": str(error)}, indent=2))
            print("Partial results saved:", archive_results())
            raise
        results.extend(rows)
        selected_parameters.append(selection)
        pd.DataFrame(results).to_csv(RUN_DIR / "fold_scores.csv", index=False)
        pd.DataFrame(selected_parameters).to_csv(RUN_DIR / "selected_parameters.csv", index=False)
        archive_results()


Fold 1: 13441 clean training; 1494 held-out faces
  Inner 1/3: fitting train-only scaler/PCA64
  Inner 2/3: fitting train-only scaler/PCA64
  Inner 3/3: fitting train-only scaler/PCA64
  Refit selected parameters on all outer-training faces
  Learning curve inner 1, fraction 0.1: 897 clean training faces
  Learning curve inner 1, fraction 0.25: 2241 clean training faces
  Learning curve inner 1, fraction 0.5: 4481 clean training faces
  Learning curve inner 1, fraction 0.75: 6722 clean training faces
  Learning curve inner 1, fraction 1: 8960 clean training faces
  Learning curve inner 2, fraction 0.1: 897 clean training faces
  Learning curve inner 2, fraction 0.25: 2241 clean training faces
  Learning curve inner 2, fraction 0.5: 4482 clean training faces
  Learning curve inner 2, fraction 0.75: 6723 clean training faces
  Learning curve inner 2, fraction 1: 8961 clean training faces
  Learning curve inner 3, fraction 0.1: 897 clean training faces
  Learning curve inner 3, fraction 0

## 18. Summarize metrics and combine test predictions

Combine test predictions and report both mean/SD across folds and separate pooled metrics. A single-fold SD is undefined; QUICK-test results are not final performance estimates.


In [150]:
scores = pd.DataFrame(results)
metric_columns = [name for name in scores.columns if name not in ("fold", "condition", "n_test", "n_samples")]
summary = scores.groupby("condition")[metric_columns].agg(["mean", "std"]).reindex(CONDITIONS)
summary.to_csv(RUN_DIR / "summary.csv")
all_predictions = pd.concat([
    pd.read_csv(path) for path in sorted(RUN_DIR.glob("fold_*/test/*/predictions.csv"))
], ignore_index=True)
all_predictions.to_csv(RUN_DIR / "all_test_predictions.csv", index=False)
for condition, table in all_predictions.groupby("condition"):
    if table["metadata_row"].duplicated().any():
        raise ValueError("An image appears twice in pooled outer-test results.")
    save_metric_reports(table["y_true"], table["y_pred"], RUN_DIR / "pooled_test" / condition)
fold_f1 = scores.pivot(index="fold", columns="condition", values="macro_f1")
f1_drop = pd.DataFrame({condition: fold_f1["clean"] - fold_f1[condition] for condition in CONDITIONS[1:]})
f1_drop.to_csv(RUN_DIR / "paired_macro_f1_drop.csv")
print("QUICK TEST ONLY" if QUICK_TEST else ("10-FOLD RESULTS" if scores["fold"].nunique() == 10 else "PARTIAL OUTER-FOLD RESULTS"))
display(summary.round(4))
display(f1_drop.round(4))


10-FOLD RESULTS


accuracy         balanced_accuracy         macro_precision  \
                   mean     std              mean     std            mean   
condition                                                                   
clean            0.2914  0.0106            0.2725  0.0109          0.2731   
upper_mask       0.2546  0.0122            0.2197  0.0119          0.2606   
lower_mask       0.2044  0.0117            0.1862  0.0137          0.1898   
gaussian_blur    0.2909  0.0108            0.2722  0.0110          0.2730   
gaussian_noise   0.2918  0.0106            0.2733  0.0101          0.2738   

                       macro_recall         macro_f1          ...  \
                   std         mean     std     mean     std  ...   
condition                                                     ...   
clean           0.0120       0.2725  0.0109   0.2694  0.0102  ...   
upper_mask      0.0257       0.2197  0.0119   0.2024  0.0116  ...   
lower_mask      0.0158       0.1862  0.0137   0.1681  0.0150  ...   
gaussian_blur   0.0121       0.2722  0.0110   0.2691  0.0103  ...   
gaussian_noise  0.0111       0.2733  0.0101   0.2700  0.0093  ...   

               weighted_recall         weighted_f1         micro_precision  \
                          mean     std        mean     std            mean   
condition                                                                    
clean                   0.2914  0.0106      0.2939  0.0114          0.2914   
upper_mask              0.2546  0.0122      0.2300  0.0113          0.2546   
lower_mask              0.2044  0.0117      0.1836  0.0132          0.2044   
gaussian_blur           0.2909  0.0108      0.2935  0.0117          0.2909   
gaussian_noise          0.2918  0.0106      0.2941  0.0113          0.2918   

                       micro_recall         micro_f1          
                   std         mean     std     mean     std  
condition                                                     
clean           0.0106       0.2914  0.0106   0.2914  0.0106  
upper_mask      0.0122       0.2546  0.0122   0.2546  0.0122  
lower_mask      0.0117       0.2044  0.0117   0.2044  0.0117  
gaussian_blur   0.0108       0.2909  0.0108   0.2909  0.0108  
gaussian_noise  0.0106       0.2918  0.0106   0.2918  0.0106  

[5 rows x 22 columns]

,upper_mask,lower_mask,gaussian_blur,gaussian_noise
fold,,,,
1,0.0789,0.0940,-0.0021,-0.0007
2,0.0438,0.0962,-0.0015,0.0022
3,0.0495,0.0858,0.0005,-0.0025
4,0.0658,0.1128,0.0017,0.0026
5,0.0718,0.1272,-0.0005,-0.0011
6,0.0593,0.1044,0.0035,-0.0013
7,0.0715,0.0886,-0.0013,-0.0035
8,0.0672,0.0999,0.0010,-0.0039
9,0.0878,0.1090,0.0006,0.0012


## 19. Compare the image conditions in a table

Each row is one image condition. Columns show accuracy, macro precision, macro recall, macro F1, weighted F1 and the paired macro-F1 drop from the original images. Macro metrics weight classes equally; weighted F1 weights them by their test support. Balanced accuracy duplicates macro recall when all classes are represented, and micro F1 equals accuracy here, so these are not repeated.

Display mean ± sample SD across completed outer folds, or just the value for one fold. All scores are on a 0–1 scale except the F1 drop, which can be negative: positive means degradation; negative means improvement. Save full-precision means/SDs as CSV and the displayed table as HTML. If results are already available in memory, rerun this cell and step 20; no model refitting is needed.


In [151]:
condition_names = {
    "clean": "Original images",
    "upper_mask": "Upper-half mask",
    "lower_mask": "Lower-half mask",
    "gaussian_blur": "Gaussian blur (sigma=1)",
    "gaussian_noise": "Gaussian noise (sd=0.05)",
}
metric_names = {
    "accuracy": "Accuracy",
    "macro_precision": "Macro precision",
    "macro_recall": "Macro recall",
    "macro_f1": "Macro F1",
    "weighted_f1": "Weighted F1",
    "macro_f1_drop": "Macro F1 drop",
}

# Compute the change against the original images within each matched fold.
comparison_scores = scores.copy()
original_f1 = scores.loc[scores["condition"] == "clean"].set_index("fold")["macro_f1"]
comparison_scores["macro_f1_drop"] = (
    comparison_scores["fold"].map(original_f1) - comparison_scores["macro_f1"]
)
grouped = comparison_scores.groupby("condition")[list(metric_names)]
comparison_mean = grouped.mean().reindex(CONDITIONS)
comparison_sd = grouped.std().reindex(CONDITIONS)
comparison_mean = comparison_mean.rename(index=condition_names, columns=metric_names)
comparison_sd = comparison_sd.rename(index=condition_names, columns=metric_names)
comparison_mean.index.name = comparison_sd.index.name = "Data"

# Keep full precision in the numeric files; round only the displayed table.
comparison_mean.to_csv(RUN_DIR / "comparison_metrics_mean.csv")
comparison_sd.to_csv(RUN_DIR / "comparison_metrics_sd.csv")
comparison_table = comparison_mean.copy().astype(object)
for column in comparison_table.columns:
    comparison_table[column] = [
        f"{mean:.4f}" if pd.isna(sd) else f"{mean:.4f} ± {sd:.4f}"
        for mean, sd in zip(comparison_mean[column], comparison_sd[column])
    ]

n_folds = scores["fold"].nunique()
print("QUICK TEST ONLY" if QUICK_TEST else "Held-out test results")
print(f"{n_folds} completed outer fold(s); values are mean ± SD when multiple folds are available.")
print("Higher is better, except Macro F1 drop: positive means worse than the original images.")
display(comparison_table)
comparison_table.to_html(RUN_DIR / "comparison_table.html")


Held-out test results
10 completed outer fold(s); values are mean ± SD when multiple folds are available.
Higher is better, except Macro F1 drop: positive means worse than the original images.


,Accuracy,Macro precision,Macro recall,Macro F1,Weighted F1,Macro F1 drop
Data,,,,,,
Original images,0.2914 ± 0.0106,0.2731 ± 0.0120,0.2725 ± 0.0109,0.2694 ± 0.0102,0.2939 ± 0.0114,0.0000 ± 0.0000
Upper-half mask,0.2546 ± 0.0122,0.2606 ± 0.0257,0.2197 ± 0.0119,0.2024 ± 0.0116,0.2300 ± 0.0113,0.0669 ± 0.0132
Lower-half mask,0.2044 ± 0.0117,0.1898 ± 0.0158,0.1862 ± 0.0137,0.1681 ± 0.0150,0.1836 ± 0.0132,0.1013 ± 0.0125
Gaussian blur (sigma=1),0.2909 ± 0.0108,0.2730 ± 0.0121,0.2722 ± 0.0110,0.2691 ± 0.0103,0.2935 ± 0.0117,0.0002 ± 0.0017
Gaussian noise (sd=0.05),0.2918 ± 0.0106,0.2738 ± 0.0111,0.2733 ± 0.0101,0.2700 ± 0.0093,0.2941 ± 0.0113,-0.0007 ± 0.0022


In [152]:
CURVE_METRIC = "macro_f1"  # Change to "accuracy" to plot accuracy instead.
curve_labels = {
    "clean": "Original images",
    "upper_mask": "Upper-half mask",
    "lower_mask": "Lower-half mask",
    "gaussian_blur": "Gaussian blur",
    "gaussian_noise": "Gaussian noise",
}
curve_files = sorted(RUN_DIR.glob("fold_*/learning_curves/results.csv"))
curve_columns = [f"validation_{condition}_{CURVE_METRIC}" for condition in CONDITIONS]

if not curve_files:
    print("No learning-curve results found. Set RUN_LEARNING_CURVES=True and run the experiment.")
else:
    curve_data = pd.concat([pd.read_csv(path) for path in curve_files], ignore_index=True)
    curve_data = curve_data.loc[curve_data["status"] == "complete"]
    if not set(curve_columns).issubset(curve_data.columns):
        print("Five-condition learning curves are unavailable. Run the updated learning-curve code; "
              "older results contain only clean validation images, or all points were skipped.")
    else:
        # Keep matched points with results for every condition, not old clean-only rows.
        curve_data = curve_data.dropna(subset=curve_columns)
        if curve_data.empty:
            print("No complete five-condition learning-curve points are available.")
        else:
            # Average inner folds first, then give each available outer fold equal weight.
            curve_by_fold = curve_data.groupby(["outer_fold", "training_fraction"])[
                ["n_train"] + curve_columns].mean()
            curve_summary = curve_by_fold.groupby("training_fraction").mean()
            curve_summary["n_outer_folds"] = curve_by_fold.groupby("training_fraction").size()
            curve_summary["n_inner_points"] = curve_data.groupby("training_fraction").size()
            curve_summary.to_csv(RUN_DIR / f"learning_curves_by_condition_{CURVE_METRIC}.csv")

            figure, axis = plt.subplots(figsize=(7, 4))
            for condition in CONDITIONS:
                axis.plot(curve_summary["n_train"],
                          curve_summary[f"validation_{condition}_{CURVE_METRIC}"],
                          marker="o", label=curve_labels[condition])
            axis.set(xlabel="Clean training images (mean across folds)",
                     ylabel="Validation " + CURVE_METRIC.replace("_", " "), ylim=(0, 1),
                     title="Learning curves by image condition")
            axis.legend()
            figure.tight_layout()
            figure.savefig(RUN_DIR / f"learning_curves_by_condition_{CURVE_METRIC}.png", dpi=160)
            display(figure)
            plt.close(figure)
            print("Diagnostic validation curves, not independent outer-test performance.")

Five-condition learning curves are unavailable. Run the updated learning-curve code; older results contain only clean validation images, or all points were skipped.


In [153]:
print("Results saved in:", RUN_DIR)

Results saved in: /Users/fr/Desktop/COMP90051/expw_svm_results/rgb224_pca64_rbf_xjgu08q6
